# 09 - LSTM Neural Networks for RUL Estimation
### Phase 3: Deep Learning Model Development, Training & Evaluation

This notebook investigates **Long Short-Term Memory (LSTM)** recurrent architectures for Remaining Useful Life (RUL) estimation on multivariate temporal sensor sequences from **NASA C-MAPSS FD001**.

#### Key Objectives:
1. Load 3D multivariate temporal sliding windows $(N, L, D)$ with zero leakage across engine boundaries.
2. Formulate stacked LSTM regression networks with Adam optimization and Huber loss.
3. Evaluate RUL predictions against the official NASA test fleet terminal cycle benchmark ($N=100$).
4. Analyze sensitivity to temporal window lengths ($L \in \{30, 50, 60\}$).
5. Track continuous degradation trajectories across operational cycles.


In [ ]:
import sys
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(Path("..").resolve() / "src"))

from deep_learning.dataset_loader import get_cmapss_dataloaders
from deep_learning.lstm_model import LSTMRULModel
from deep_learning.evaluate import evaluate_cmapss_rul

print("PyTorch version:", torch.__version__)
print("Device available:", "CUDA" if torch.cuda.is_available() else "CPU")


## 1. Load FD001 Sequence Data and Model Weights


In [ ]:
# Load FD001 (w=30) sequences
loaders_w30 = get_cmapss_dataloaders("FD001", window_length=30, batch_size=256)
in_dim = loaders_w30["num_features"]

print(f"Sequence Length (L): {loaders_w30['sequence_length']}")
print(f"Number of Sensor/Operational Features: {in_dim}")
print(f"Train Sequences: {len(loaders_w30['train_data']['X'])}")
print(f"Validation Sequences: {len(loaders_w30['val_data']['X'])}")
print(f"Test Sequences (All Windows): {len(loaders_w30['test_data']['X'])}")
print(f"Official Test Engines (Terminal Cycles): {len(loaders_w30['terminal_test_indices'])}")


In [ ]:
# Instantiate LSTM Architecture and Load Checkpoint
lstm_model = LSTMRULModel(input_dim=in_dim, hidden_dim=32, num_layers=1, fc_dim=32)
checkpoint_path = Path("../models/deep_learning/lstm/nasa_rul_lstm_FD001_w30.pt")
meta_path = Path("../models/deep_learning/lstm/nasa_rul_lstm_FD001_w30_meta.json")

lstm_model.load_state_dict(torch.load(checkpoint_path, map_location="cpu", weights_only=True))
lstm_model.eval()

with open(meta_path, "r") as f:
    meta = json.load(f)

print("LSTM Architecture:", meta["config"])
print(f"Total Parameters: {meta['size']['total_parameters']:,}")
print(f"Best Training Epoch: {meta['train']['best_epoch']}")


## 2. Benchmark Evaluation on Official NASA Test Fleet


In [ ]:
eval_results = evaluate_cmapss_rul(
    lstm_model,
    loaders_w30["val_loader"],
    loaders_w30["test_loader"],
    loaders_w30["val_data"]["y_rul"],
    loaders_w30["test_data"]["y_rul"],
    loaders_w30["terminal_test_indices"],
    device="cpu"
)

term_metrics = eval_results["test_terminal_metrics"]
print("=" * 60)
print("NASA C-MAPSS FD001 LSTM TEST FLEET BENCHMARK (N=100)")
print("=" * 60)
print(f"  Test Terminal RMSE:       {term_metrics['test_terminal_rmse']:.2f} cycles")
print(f"  Test Terminal MAE:        {term_metrics['test_terminal_mae']:.2f} cycles")
print(f"  Test Terminal NASA Score: {term_metrics['test_terminal_nasa_score']:.1f}")
print(f"  Test Terminal R2 Score:   {term_metrics['test_terminal_r2']:.4f}")
print("=" * 60)


## 3. Predicted vs Actual RUL Scatter and Residual Analysis


In [ ]:
y_true = eval_results["y_test_terminal_true"]
y_pred = eval_results["y_test_terminal_pred"]
errors = y_pred - y_true

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
max_val = max(float(np.max(y_true)), float(np.max(y_pred))) + 15
ax1.scatter(y_true, y_pred, color="#1f77b4", edgecolors="k", alpha=0.8, s=50)
ax1.plot([0, max_val], [0, max_val], "r--", linewidth=2, label="Identity line (ideal)")
ax1.set_title("LSTM RUL: Predicted vs Actual (Test Fleet N=100)", fontweight="bold")
ax1.set_xlabel("Official True RUL (Cycles)")
ax1.set_ylabel("Predicted RUL (Cycles)")
ax1.legend(loc="upper left")
ax1.grid(True, linestyle=":", alpha=0.6)

# Residual error distribution
ax2.hist(errors, bins=20, color="#1f77b4", edgecolor="black", alpha=0.7)
ax2.axvline(0, color="red", linestyle="--", linewidth=2, label="Zero Error")
ax2.set_title(f"Residual Distribution (Mean: {errors.mean():.1f}, Std: {errors.std():.1f})", fontweight="bold")
ax2.set_xlabel("Error (Pred - True) in Cycles")
ax2.set_ylabel("Engine Count")
ax2.legend(loc="upper right")
ax2.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()


## 4. Multi-Cycle Degradation Trajectory Tracking


In [ ]:
# Select test engines to track continuous degradation across cycles
test_uids = loaders_w30["test_data"]["engine_ids"]
test_cycles = loaders_w30["test_data"]["end_cycles"]
test_all_true = loaders_w30["test_data"]["y_rul"]
test_all_pred = eval_results["y_test_all_pred"]

sample_uids = [31, 49, 81]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

for ax, uid in zip(axes, sample_uids):
    mask = (test_uids == uid)
    ax.plot(test_cycles[mask], test_all_true[mask], "k--", linewidth=2.2, label="True RUL")
    ax.plot(test_cycles[mask], test_all_pred[mask], color="#1f77b4", linewidth=2, label="LSTM Pred")
    ax.set_title(f"Test Engine #{uid} Degradation", fontweight="bold")
    ax.set_xlabel("Operating Cycle")
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.legend(loc="upper right")

axes[0].set_ylabel("Remaining Useful Life (Cycles)")
plt.tight_layout()
plt.show()
